In [1]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')

In [2]:

from sklearn.preprocessing import  LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, confusion_matrix
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

In [3]:

df = pd.read_csv("project3_market_segmentation.csv")

df.head()

,Customer_ID,Age,Distance_From_Market_km,Monthly_Spending_XAF,Visit_Frequency_Monthly,Category_Preference,Payment_Method,Items_Per_Visit,Years_As_Customer,True_Segment
0,CUST-00001,27,2.1,21260,2.5,Tools,Cash,1,2,Occasional Shopper
1,CUST-00002,33,7.3,122636,11.2,Clothing,Mobile Money,21,1,Regular Shopper
2,CUST-00003,52,0.4,479683,9.9,Food,Mobile Money,40,11,Wholesale Buyer
3,CUST-00004,25,10.8,66261,7.7,Clothing,Mobile Money,15,7,Regular Shopper
4,CUST-00005,50,29.9,103394,3.0,Electronics,Cash,16,1,Regular Shopper


In [4]:
df.info ()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Customer_ID              10000 non-null  object 
 1   Age                      10000 non-null  int64  
 2   Distance_From_Market_km  10000 non-null  float64
 3   Monthly_Spending_XAF     10000 non-null  int64  
 4   Visit_Frequency_Monthly  10000 non-null  float64
 5   Category_Preference      10000 non-null  object 
 6   Payment_Method           10000 non-null  object 
 7   Items_Per_Visit          10000 non-null  int64  
 8   Years_As_Customer        10000 non-null  int64  
 9   True_Segment             10000 non-null  object 
dtypes: float64(2), int64(4), object(4)
memory usage: 781.4+ KB


,Age,Distance_From_Market_km,Monthly_Spending_XAF,Visit_Frequency_Monthly,Items_Per_Visit,Years_As_Customer
count,10000.0000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,32.3228,5.215880,137485.484300,9.646550,24.153400,7.472000
std,9.3563,5.048337,194839.993487,7.073915,17.373614,4.033986
min,18.0000,0.200000,1800.000000,0.500000,1.000000,1.000000
25%,25.0000,1.700000,24219.500000,3.000000,9.000000,4.000000
50%,32.0000,3.700000,49552.500000,8.100000,20.000000,8.000000
75%,39.0000,7.100000,106653.000000,15.200000,38.000000,11.000000
max,70.0000,40.000000,997866.000000,35.000000,80.000000,14.000000


In [5]:
data = df.drop(["Customer_ID", "True_Segment"], axis=1)

In [6]:
le = LabelEncoder()

data["Category_Preference"] = le.fit_transform(data["Category_Preference"])
data["Payment_Method"] = le.fit_transform(data["Payment_Method"])

In [7]:
data.head()

,Age,Distance_From_Market_km,Monthly_Spending_XAF,Visit_Frequency_Monthly,Category_Preference,Payment_Method,Items_Per_Visit,Years_As_Customer
0,27,2.1,21260,2.5,4,0,1,2
1,33,7.3,122636,11.2,0,2,21,1
2,52,0.4,479683,9.9,2,2,40,11
3,25,10.8,66261,7.7,0,2,15,7
4,50,29.9,103394,3.0,1,0,16,1


In [8]:
scaler=StandardScaler()
scaled_data=scaler.fit_transform(data)

In [9]:
wcss=[]
for i in range (1,11):
    kmeans=KMeans(n_clusters=i, random_state=42)
    kmeans.fit(scaled_data)
    wcss.append(kmeans.inertia_)
plt.plot(np.arange(1,11),wcss)
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")
plt.savefig("elbow_plot.png")
plt.show()


C:\Users\user\AppData\Local\Temp\ipykernel_10356\2328003105.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
for i,inertia in zip(np.arange(1,11),wcss):
    print(f"| {i} || --> |{inertia} || gap = {-wcss[(i +1 )% 10]+wcss[i%10]} \n")

| 1 || --> |80000.00000000003 || gap = 4975.154425462526 

| 2 || --> |60688.81610717464 || gap = 7566.318431162719 

| 3 || --> |55713.66168171212 || gap = 3731.455525726531 

| 4 || --> |48147.3432505494 || gap = 1119.7641396360705 

| 5 || --> |44415.88772482287 || gap = 2960.660452013006 

| 6 || --> |43296.1235851868 || gap = 1480.4740319632547 

| 7 || --> |40335.46313317379 || gap = 2705.884473915052 

| 8 || --> |38854.989101210536 || gap = 1414.6910515089257 

| 9 || --> |36149.104627295485 || gap = -45265.58642421347 

| 10 || --> |34734.41357578656 || gap = 19311.183892825386 



In [11]:
Kmeans=KMeans(n_clusters=4,random_state=42)
clusters=Kmeans.fit_predict(scaled_data)
df["cluster"] = clusters
df["True_Segment"].unique()
df.head(10)


,Customer_ID,Age,Distance_From_Market_km,Monthly_Spending_XAF,Visit_Frequency_Monthly,Category_Preference,Payment_Method,Items_Per_Visit,Years_As_Customer,True_Segment,cluster
0,CUST-00001,27,2.1,21260,2.5,Tools,Cash,1,2,Occasional Shopper,2
1,CUST-00002,33,7.3,122636,11.2,Clothing,Mobile Money,21,1,Regular Shopper,1
2,CUST-00003,52,0.4,479683,9.9,Food,Mobile Money,40,11,Wholesale Buyer,0
3,CUST-00004,25,10.8,66261,7.7,Clothing,Mobile Money,15,7,Regular Shopper,1
4,CUST-00005,50,29.9,103394,3.0,Electronics,Cash,16,1,Regular Shopper,1
5,CUST-00006,24,2.4,15511,2.8,Electronics,Cash,1,9,Occasional Shopper,1
6,CUST-00007,31,6.6,739021,9.6,Tools,Cash,32,7,Wholesale Buyer,0
7,CUST-00008,47,2.2,50506,7.5,Clothing,Mobile Money,17,7,Regular Shopper,1
8,CUST-00009,18,18.7,22843,1.4,Tools,Mobile Money,3,10,Occasional Shopper,2
9,CUST-00010,44,5.2,566410,17.7,Food,Cash,38,5,Wholesale Buyer,0


In [12]:
pca=PCA(n_components=2)
pca_data= pca.fit_transform(scaled_data)
df["PCA1"]= pca_data[:,0]
df["PCA2"]= pca_data [:,1]

In [13]:
plt.figure(figsize=(8,6))
sns.scatterplot(
    x=df["PCA1"],
    y=df["PCA2"],
    hue =df["cluster"],
    palette="Set2"
)
plt.title("Custumer Segmentation Clusters")
plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_10356\3247386530.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
plt.savefig("cluster_plot.png")

In [15]:
pd.crosstab(df["cluster"], df["True_Segment"])

True_Segment,Loyal Budget Buyer,Occasional Shopper,Regular Shopper,Wholesale Buyer
cluster,,,,
0,0,0,1,1911
1,0,1855,1410,6
2,8,645,2089,23
3,1992,0,0,60


In [16]:
sns.boxplot(x=df["cluster"], y=df["Monthly_Spending_XAF"], data=df)
plt.title("Spending by CLuster")
plt.savefig("spending_by_cluster.png")
plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_10356\3581714900.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
joblib.dump(Kmeans,"customer_segmentation_model.joblib")

['customer_segmentation_model.joblib']

In [18]:
joblib.dump(scaler,"scaler.joblib")

['scaler.joblib']

In [19]:
df.to_csv("segmented_customers.csv", index=False)
df.head()

,Customer_ID,Age,Distance_From_Market_km,Monthly_Spending_XAF,Visit_Frequency_Monthly,Category_Preference,Payment_Method,Items_Per_Visit,Years_As_Customer,True_Segment,cluster,PCA1,PCA2
0,CUST-00001,27,2.1,21260,2.5,Tools,Cash,1,2,Occasional Shopper,2,-1.303051,0.443069
1,CUST-00002,33,7.3,122636,11.2,Clothing,Mobile Money,21,1,Regular Shopper,1,-0.377881,0.573002
2,CUST-00003,52,0.4,479683,9.9,Food,Mobile Money,40,11,Wholesale Buyer,0,1.862849,1.699236
3,CUST-00004,25,10.8,66261,7.7,Clothing,Mobile Money,15,7,Regular Shopper,1,-1.268161,0.258462
4,CUST-00005,50,29.9,103394,3.0,Electronics,Cash,16,1,Regular Shopper,1,-0.232463,-0.293395


In [20]:
df["cluster"].value_counts()

cluster
1    3271
2    2765
3    2052
0    1912
Name: count, dtype: int64

In [21]:
import joblib
joblib.dump(data.columns.tolist(),"columns.joblib")

['columns.joblib']